# **Kaggle competition -  Home Credit Risk Prediction - H2O train**

This is my first Kaggle competition. While I have worked on datasets through the MIT Professional course, this is the first real world dataset. This dataset is complex and huge, hence I plan to take a systematic step by step approach. Understanding the dataset is of utmost importance for a successful data scientist. A good insight into data will help me make better decisions aboout the aggregation I would like to make and any feature engineering once I have a hanlde of all data and features I have used to achieve best possible results. Hence, I will be taking a slow and incremental change approach. This will not only make tracing changes easier, but also enhance my learning by letting me better understand what step leads to what change.

I have learned a lot from fellow Kagglers who are gracious in sharing their code as well as their knowledge. I have extensively refered to some of the following notebooks for my learning process.

Reference files for this is:

https://www.kaggle.com/code/dksdms4/lb-0-565-improved-baseline-notebook

https://www.kaggle.com/code/greysky/home-credit-baseline

https://www.kaggle.com/code/ravi20076/homecredit-starter-inference-v1

https://www.kaggle.com/code/peizhengwang/lb-0-57-mod-weight-pure-lgb

### **Version information**

- This is version 5.
- This is the first subversion of version 5. Includes all depth 1 tables

In this version,
- All depth 1 tables included
- dates are all changed to months except age which is changes to days
- incoming and outcoming amts calculated in other table
- date columns of taxregistry_a, taxregistry_b, taxregistry_c dropped
- Final filtering before split

In [ ]:
#!pip install h2o4gpu

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing

# import further libraries
import matplotlib.pyplot as plt
import seaborn as sns

# library for cross validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

# library for metrics
from sklearn.metrics import roc_auc_score

# library for LightGBM Model
#import lightgbm as lgbm
#from lightgbm import LGBMClassifier

# library for MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# library for optuna
import optuna

# Importing necessary packages 
import h2o
from h2o.automl import H2OAutoML

# library to read and write files
import pickle

# library to save and load models
import joblib

# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# library for garbage collection
import gc  # since the data is huge here, regularly cleaning up will free up memory

# library to catch and ignore warnings
import warnings
warnings.filterwarnings("ignore")

# library for utility script containing various pipelines
import homecreditutility_v4 as hcu #version 21

## **Data Preparation**

We have data from different sources and they are split into 3 depths.
- Depth 0 has base data and static data from both internal and external sources
- Depth 1 has data from both internal and expternal sources that changes over time. We will need to use aggregate the values to use this
- Depth 2 has similar data as depth 1 and we will need to aggregate the values to use this.

Since, there is data available in 3 different depths, we will do the necessary processing in a systematic manner by considering each depth at a time. Once we have downloaded the data, combined into a single dataframe, cleaned the data, feature engineered and made it ready for the model, then we will move onto modeling.

#### **Loading data onto dataframes**

### **H2O initialization**

In [ ]:
# Initialize H2O
h2o.init()

In [ ]:
%%time
train_hf = h2o.import_file("/kaggle/input/v7-data-prep-homecreditrisk2024/train_df.csv")

In [ ]:
# splitting dataset into train and test sets
#splits = train_hf.split_frame(ratios=[0.8])
#train_aml = splits[0]
#valid_aml = splits[1]

# obtain the features and target for modelling.
y_feature = 'target'  #insert name of the target column here
x_feature = list(train_hf.columns)
x_feature.remove('target')
x_feature.remove('case_id')
x_feature.remove('WEEK_NUM')

In [ ]:
# convert target column to categorical in h2o
#train_aml[y_feature] = train_aml[y_feature].asfactor()
#valid_aml[y_feature] = valid_aml[y_feature].asfactor()
train_hf[y_feature] = train_hf[y_feature].asfactor()

In [ ]:
aml = H2OAutoML(
    max_models = 11, 
    max_runtime_secs = 14400,
    max_runtime_secs_per_model = 1200,
    seed = 13,
    include_algos = ["GBM", "StackedEnsemble"], 
    verbosity = 'info', 
    stopping_metric='auc')

In [ ]:
%%time
aml.train(x = x_feature, y = y_feature, training_frame = train_hf) #, validation_frame = valid_aml

In [ ]:
aml.leaderboard

In [ ]:
# save all H2o models
for i, mod_id in enumerate(aml.leaderboard):
    aml1 = h2o.get_model(aml.leaderboard[i,0])  # Get model object
    h2o.save_model(model=aml1, path="./h2omodels/")

## **Function to determine gini_stability**

In [ ]:
def gini_stability(base, w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", "predict"]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", "predict"]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x["predict"])-1).tolist()

    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std

### **H2O leader prediction and stability score**

In [ ]:
%%time
# prediction with the best performer from h2o
y_pred = aml.leader.predict(train_hf[x_feature])
y_pred.head()

In [ ]:
%%time
# converst H2o fromes to pandas
y_true = h2o.as_list(train_hf['target'])
y_pred = h2o.as_list(y_pred[:,2])

# display AUC score for the train and validation datasets
print(f'The AUC score on the train set is: {roc_auc_score(y_true, y_pred)}')

In [ ]:
%%time
pred_df = h2o.as_list(train_hf[["WEEK_NUM", "target"]])
pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
print(f'The stability score on the train set is: {stability_score_train}')

### **H2O second best model prediction and stability score**

In [ ]:
# load the model from model path
model_path = "/kaggle/working/h2omodels/" + aml.leaderboard[1, 0] 
model = h2o.load_model(model_path)

In [ ]:
# prediction with the second best performer from h2o
y_pred = model.predict(train_hf[x_feature])
y_pred.head()

In [ ]:
%%time
# convert H2o fromes to pandas
y_true = h2o.as_list(train_hf['target'])
y_pred = h2o.as_list(y_pred[:,2])

# display AUC score for the train and validation datasets
print(f'The AUC score on the train set is: {roc_auc_score(y_true, y_pred)}')

In [ ]:
%%time
pred_df = h2o.as_list(train_hf[["WEEK_NUM", "target"]])
pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
print(f'The stability score on the train set is: {stability_score_train}')

### **H2O third best model prediction and stability score**

In [ ]:
# load the model from model path
model_path = "/kaggle/working/h2omodels/" + aml.leaderboard[2, 0] 
model = h2o.load_model(model_path)

In [ ]:
# prediction with the second best performer from h2o
y_pred = model.predict(train_hf[x_feature])
y_pred.head()

In [ ]:
%%time
# convert H2o fromes to pandas
y_true = h2o.as_list(train_hf['target'])
y_pred = h2o.as_list(y_pred[:,2])

# display AUC score for the train and validation datasets
print(f'The AUC score on the train set is: {roc_auc_score(y_true, y_pred)}')

In [ ]:
%%time
pred_df = h2o.as_list(train_hf[["WEEK_NUM", "target"]])
pred_df["predict"] = y_pred

# finding the stability scores for train and validation datasets
stability_score_train = gini_stability(pred_df)

# display the stability scores for train and validation dataset
print(f'The stability score on the train set is: {stability_score_train}')